In [2]:
!pip install ucimlrepo kagglehub --quiet


In [3]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import tensorflow as tf


In [4]:
retention_levels = [0.10, 0.25, 0.50, 0.75, 0.90]


In [5]:
def pca_classification_accuracy(X, y, retention_levels):
    """
    Computes baseline and PCA-reduced classification accuracy
    using an 80/20 stratified split and kNN classifier.
    """
    results = []

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Baseline (no PCA)
    baseline_model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ])

    baseline_model.fit(X_train, y_train)
    baseline_acc = accuracy_score(y_test, baseline_model.predict(X_test))

    original_dims = X.shape[1]

    for r in retention_levels:
        n_components = max(1, int(original_dims * r))

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=n_components)),
            ('knn', KNeighborsClassifier(n_neighbors=5))
        ])

        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))

        results.append({
            'Retention Level (%)': int(r * 100),
            'PCA Components': n_components,
            'Accuracy (%)': acc * 100
        })

    return baseline_acc * 100, pd.DataFrame(results)


In [6]:
wine_data = fetch_ucirepo(id=186)
X_wine = wine_data.data.features.values
y_wine = wine_data.data.targets.values.ravel()

baseline_wine, wine_acc = pca_classification_accuracy(
    X_wine, y_wine, retention_levels
)

wine_acc['Dataset'] = 'Wine'
wine_acc['Method'] = 'PCA'
wine_acc['Baseline (%)'] = baseline_wine

wine_acc


,Retention Level (%),PCA Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,1,44.384615,Wine,PCA,55.846154
1,25,2,46.461538,Wine,PCA,55.846154
2,50,5,53.384615,Wine,PCA,55.846154
3,75,8,55.692308,Wine,PCA,55.846154
4,90,9,57.076923,Wine,PCA,55.846154


In [7]:
breast_data = fetch_ucirepo(id=17)
X_breast = breast_data.data.features.values
y_breast = breast_data.data.targets.values.ravel()

baseline_breast, breast_acc = pca_classification_accuracy(
    X_breast, y_breast, retention_levels
)

breast_acc['Dataset'] = 'Breast Cancer'
breast_acc['Method'] = 'PCA'
breast_acc['Baseline (%)'] = baseline_breast

breast_acc


,Retention Level (%),PCA Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,3,94.736842,Breast Cancer,PCA,95.614035
1,25,7,95.614035,Breast Cancer,PCA,95.614035
2,50,15,95.614035,Breast Cancer,PCA,95.614035
3,75,22,95.614035,Breast Cancer,PCA,95.614035
4,90,27,95.614035,Breast Cancer,PCA,95.614035


In [8]:
(X_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

X_mnist = X_train.reshape(X_train.shape[0], -1)
y_mnist = y_train

baseline_mnist, mnist_acc = pca_classification_accuracy(
    X_mnist, y_mnist, retention_levels
)

mnist_acc['Dataset'] = 'MNIST'
mnist_acc['Method'] = 'PCA'
mnist_acc['Baseline (%)'] = baseline_mnist

mnist_acc


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


,Retention Level (%),PCA Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,78,95.808333,MNIST,PCA,94.783333
1,25,196,95.591667,MNIST,PCA,94.783333
2,50,392,94.958333,MNIST,PCA,94.783333
3,75,588,94.841667,MNIST,PCA,94.783333
4,90,705,94.783333,MNIST,PCA,94.783333


In [9]:
table_3 = pd.concat([wine_acc, breast_acc, mnist_acc], ignore_index=True)

table_3 = table_3[
    ['Dataset', 'Method', 'Baseline (%)',
     'Retention Level (%)', 'PCA Components', 'Accuracy (%)']
]

table_3


,Dataset,Method,Baseline (%),Retention Level (%),PCA Components,Accuracy (%)
0,Wine,PCA,55.846154,10,1,44.384615
1,Wine,PCA,55.846154,25,2,46.461538
2,Wine,PCA,55.846154,50,5,53.384615
3,Wine,PCA,55.846154,75,8,55.692308
4,Wine,PCA,55.846154,90,9,57.076923
5,Breast Cancer,PCA,95.614035,10,3,94.736842
6,Breast Cancer,PCA,95.614035,25,7,95.614035
7,Breast Cancer,PCA,95.614035,50,15,95.614035
8,Breast Cancer,PCA,95.614035,75,22,95.614035
9,Breast Cancer,PCA,95.614035,90,27,95.614035
